# Training 06 · Ablations of the patch loop

Seed-0 loops with one change each: detection threshold τ = 0.1 / 0.5, ranking margin 0.1 / 1.0, patching only the
regression head, and preferring the raw input instead of the codec reconstruction in each pair. About 1.3 h each.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first run); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['ablation']]
print(len(KEYS), "runs:", ", ".join(KEYS))

6 runs: abl_tau0.1_s0, abl_margin0.1_s0, abl_headonly_s0, abl_tau0.5_s0, abl_margin1.0_s0, abl_prefinput_s0


In [4]:
# Stored configuration of the first run (checkpoint paths point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: hack_loop
task: hack_loop
phase: ablation
seed: 0
run_name: hack_loop_tau0.1_s0__2026-09-15_16-12-53
init_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt
model:
  pretrained: facebook/wav2vec2-base
  freeze_feature_encoder: true
  freeze_encoder: false
data:
  corpus: bvcc
  max_seconds: 6.0
  num_workers: 8
loop:
  rounds: 3
  pair_preferred: recon
  val_attack_utts: 400
  max_train_utts: null
attack:
  codec:
    name: encodec_24khz
    quantize: false
    encodec_bandwidth: 24.0
  policy:
    hidden: 256
    layers: 4
    kernel: 5
    init_log_std: -1.0
  action_hop: 0
  budget_mode: sphere
  eps: 0.5
  budget_penalty: 0.0
  group_size: 8
  batch_size: 8
  crop_seconds: 3.0
  steps: 2000
  lr: 0.001
  ppo_epochs: 2
  clip: 0.2
  adv_eps: 0.0001
  grad_clip: 1.0
  log_every: 10
  apply:
    n_samples: 3
    batch_size: 16
    chunk_utts: 512
    max_seconds: 20.0
detect:
  tau: 0.1
  measures:
  - estoi

In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)  # outputs: {scratch}/projects/audio_assessment_hackability/retrain/<key>/
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for abl_tau0.1_s0 ->


#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_abl_tau0.1_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key abl_tau0.1_s0



## Retrained vs original

In [6]:
import json
released = repro.fetch_results()
rows = {}
for k in KEYS:
    for src, d in (("original", released / Path(reg[k]["run_dir"]).name), ("retrained", repro.RUNS / "retrain" / k)):
        f = d / "round3/done.json"
        if f.exists():
            rows[(k, src)] = pd.json_normalize(json.loads(f.read_text())).iloc[0]
df = pd.DataFrame(rows).T
df[[c for c in ["n_pairs_round", "n_pairs_buffer", "round3/train_asr", "round3/val_asr", "round3/val_mean_gain"] if c in df]] if len(df) else "no runs found"

,,n_pairs_round,n_pairs_buffer,round3/train_asr,round3/val_asr,round3/val_mean_gain
abl_tau0.1_s0,original,0,11225,0.0,0.0025,-1.608008
abl_margin0.1_s0,original,2123,7566,0.106554,0.09,0.060977
abl_headonly_s0,original,2796,10443,0.138721,0.1,0.092793
abl_tau0.5_s0,original,694,2938,0.036992,0.03,0.032266
abl_margin1.0_s0,original,1198,7551,0.05971,0.04,-0.813691
abl_prefinput_s0,original,675,6512,0.036188,0.015,-0.238086
